<a href="https://colab.research.google.com/github/felondrum/llm_driven_development_otus/blob/develop/hw2_student_version.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📚 ДЗ №2: Работа с данными для LLM

## 🎯 Цель задания
После выполнения задания вы сможете:
- Предобрабатывать русскоязычные текстовые данные для LLM
- Работать с готовыми моделями HuggingFace для анализа тональности и NER
- Создавать эффективные промпты для LLM API
- Сравнивать качество работы разных подходов к анализу текста
- Формировать датасеты в формате instruction-following для fine-tuning
- Сохранять данные в правильных форматах для обучения LLM

## 📝 Структура задания
- **Часть 1** (35% оценки): Предобработка данных и работа с готовыми моделями
- **Часть 2** (35% оценки): LLM API и prompt engineering
- **Часть 3** (20% оценки): Подготовка данных для fine-tuning LLM
- **Часть 4** (10% оценки): Сравнительный анализ и визуализация

## ⚡ Критерии оценки
- Качество предобработки данных: 25%
- Корректность работы с готовыми моделями: 20%
- Эффективность промптов для LLM: 25%
- Правильность подготовки данных для fine-tuning: 20%
- Качество сравнительного анализа: 10%


## 🔧 Установка зависимостей

Установим необходимые библиотеки для работы с данными, готовыми моделями и LLM API.


In [ ]:
%pip install pandas numpy matplotlib seaborn
%pip install transformers torch
%pip install openai>=1.0.0  # Для работы с OpenAI API
%pip install datasets
%pip install pymorphy2



In [ ]:
# Импорт необходимых библиотек
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from typing import List, Dict, Tuple
from transformers import pipeline, AutoTokenizer, AutoModelForTokenClassification
import warnings
warnings.filterwarnings('ignore')

# Настройка отображения
plt.style.use('default')
sns.set_palette("husl")
%matplotlib inline

print("Библиотеки загружены успешно!")



## 📊 Часть 1: Предобработка данных и готовые модели (35% оценки)

### Задание 1.1: Анализ "грязного" датасета

Проанализируем реалистичный датасет с типичными проблемами: опечатки, разные регистры, лишние пробелы, эмодзи.


In [ ]:
# Создаем "грязный" датасет с типичными проблемами реальных данных
# Включаем сложные случаи для демонстрации преимуществ LLM
raw_reviews = [
    # Простые случаи
    "отличный iphone 14 PRO!!!  купил в магазине  apple на тверской 😊. Камера супер",
    "УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(",

    # Сарказм и ирония (сложно для классических моделей)
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился",

    # Смешанные эмоции
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен",

    # Сложная структура предложений
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги",

    # Контекстно-зависимые случаи
    "Заказал доставку в Яндекс.Еде из ресторана Дача на Рублевке - привезли холодное, но курьер Андрей был вежливый",
    "MacBook Pro 16 работает как часы уже год, покупал в iStore на Арбате у консультанта Елены",

    # Неоднозначные случаи
    "Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был",
    "Обслуживание в банке ВТБ на Тверской оставляет желать лучшего, хотя менеджер Ольга старалась помочь",

    # Сложные именованные сущности
    "Купил новый Samsung Galaxy S24 Ultra в DNS на Ленинском проспекте, консультант Дмитрий Иванович всё объяснил",
    "Ужинал в ресторане White Rabbit на Смоленской площади - шеф-повар Владимир Мухин превзошел ожидания",

    # Опечатки и сленг
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

# TODO: Создайте DataFrame и проанализируйте проблемы в данных
# Создайте DataFrame из списка raw_reviews
# Добавьте колонку с правильными метками тональности для каждого отзыва
# Проанализируйте и выведите список проблем, которые вы видите в данных
# Подумайте: какие проблемы могут повлиять на качество анализа?

# Ваш код здесь:

# 1. Создаем DataFrame
df = pd.DataFrame(raw_reviews, columns=['review'])

# 2. Добавляем колонку с правильными метками тональности
sentiment_labels = [
    'positive',  # отличный iphone 14 PRO
    'negative',  # УЖАСНОЕ обслуживание в сбербанке
    'negative',  # Спасибо за 3 часа очереди (сарказм)
    'negative',  # замечательный сервис (сарказм)
    'positive',  # хороший телефон, доволен
    'mixed',     # ресторан красивый, но официант невнимателен
    'positive',  # в целом доволен
    'negative',  # ожидал большего
    'mixed',     # холодное, но курьер вежливый
    'positive',  # работает как часы
    'mixed',     # ну такое себе, но попкорн вкусный
    'mixed',     # оставляет желать лучшего, но менеджер старалась
    'positive',  # всё объяснил
    'positive',  # превзошел ожидания
    'positive'   # норм телек, норм чел
]

df['true_sentiment'] = sentiment_labels

# 3. Добавляем дополнительные колонки для анализа
df['review_length'] = df['review'].str.len()
df['word_count'] = df['review'].str.split().str.len()
df['char_count'] = df['review'].str.len()
df['unique_words'] = df['review'].apply(lambda x: len(set(x.split())))

# 4. Анализ проблем в данных
print("="*80)
print("АНАЛИЗ ПРОБЛЕМ В ДАННЫХ")
print("="*80)

print("\n1. СТАТИСТИКА ПО ДАННЫМ:")
print(f"   Всего отзывов: {len(df)}")
print(f"   Распределение тональности:")
print(df['true_sentiment'].value_counts())
print(f"\n   Средняя длина отзыва: {df['review_length'].mean():.0f} символов")
print(f"   Среднее количество слов: {df['word_count'].mean():.1f}")


### Задание 1.2: Очистка и нормализация данных


In [ ]:
def clean_text(text: str) -> str:
    """
    Очистка и нормализация русскоязычного текста
    """
    # TODO: Реализуйте базовую очистку текста
    # Подумайте над следующими аспектами:
    # - Как убрать эмодзи и специальные символы?
    # - Как нормализовать пробелы и отступы?
    # - Нужно ли исправлять регистр? Как?
    # - Что делать с повторяющимися знаками препинания?
    # - Как разделить слитно написанные слова (например, iPhone14)?

    # Используйте регулярные выражения (модуль re)
    # Ваш код здесь:

    pass  # Замените на ваш код

# TODO: Примените функцию очистки к данным и сравните результаты
# Создайте новую колонку с очищенными текстами
# Сравните исходные и очищенные тексты



### Задание 1.3: Использование готовых моделей HuggingFace


In [ ]:
# TODO: Загрузите готовые модели HuggingFace для анализа тональности и NER
# Исследуйте HuggingFace Hub и найдите подходящие русскоязычные модели для:
# - Анализа тональности (sentiment analysis)
# - Извлечения именованных сущностей (NER)
#
# Используйте функцию pipeline() из библиотеки transformers
# Обратите внимание на параметры модели и токенизатора

# Ваш код для загрузки моделей:

def analyze_with_huggingface(texts: List[str]) -> List[Dict]:
    """
    Анализ текстов с помощью готовых моделей HuggingFace
    """
    # TODO: Реализуйте функцию анализа
    # Для каждого текста:
    # 1. Примените модель анализа тональности
    # 2. Примените модель NER
    # 3. Соберите результаты в структурированном виде
    # 4. Верните список словарей с результатами

    pass  # Замените на ваш код

# TODO: Протестируйте модели на очищенных данных
# Проанализируйте несколько текстов (для начала возьмите 3-5)
# Выведите результаты в понятном формате
# Проанализируйте качество работы моделей



## 🤖 Часть 2: LLM API и Prompt Engineering (35% оценки)

### Задание 2.1: Создание эффективных промптов


In [ ]:
def create_prompts_for_llm() -> Dict[str, str]:
    """
    Создание базовых промптов для разных задач (один промпт на задачу)
    """
    # TODO: Создайте эффективные промпты для NER и sentiment analysis
    # Подумайте о структуре хорошего промпта:
    # - Четкое описание задачи
    # - Примеры входных и выходных данных
    # - Формат ответа (JSON, текст и т.д.)
    # - Особые требования (например, для русского языка)

    # Создайте промпты для:
    # 1. Извлечения именованных сущностей (NER)
    # 2. Анализа тональности (sentiment analysis)

    # Ваш код здесь:
    pass

# TODO: Протестируйте ваши промпты
# Выведите созданные промпты и оцените их качество

# TODO: Настройте OpenAI API
# Установите API ключ через переменные окружения
# Изучите документацию OpenAI API для Python



# TODO: Реализуйте функции для работы с OpenAI API
# Создайте функции для:
# 1. Вызова OpenAI API с промптом
# 2. Обработки ответа от API
# 3. Анализа текстов с помощью ваших промптов
#
# Подумайте о:
# - Обработке ошибок API
# - Формате запроса и ответа
# - Параметрах модели (temperature, max_tokens)
#
# Протестируйте на нескольких текстах из датасета



### Задание 2.2: Сравнение результатов HuggingFace vs LLM


In [ ]:
# TODO: Сравните результаты HuggingFace моделей с LLM на одних и тех же текстах
# Создайте сравнительный анализ:
# 1. Соберите результаты обеих подходов в структурированном виде
# 2. Сравните точность анализа тональности
# 3. Сравните качество извлечения сущностей
# 4. Проанализируйте время выполнения
# 5. Оцените простоту использования
#
# Создайте визуализации для сравнения:
# - Точность по разным метрикам
# - Время обработки
# - Количество найденных сущностей
#
# Сделайте выводы о том, когда лучше использовать каждый подход



## 📚 Часть 3: Подготовка данных для Fine-tuning LLM (20% оценки)

### Задание 3.1: Создание instruction-following датасета


In [ ]:
### Задание 2.3: Анализ сложных случаев

# Выберем специально сложные примеры для демонстрации преимуществ LLM
complex_cases = [
    "Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏",
    "iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store",
    "Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой",
    "норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции"
]

print("Анализ сложных случаев:")
print("=" * 60)

# TODO: Сравните результаты HuggingFace и OpenAI на сложных случаях
# for i, text in enumerate(complex_cases):
#     print(f"\nПример {i+1}: {text}")
#     # hf_result = sentiment_pipeline(text)
#     # openai_result = analyze_with_openai([text])
#     # print(f"HuggingFace: {hf_result}")
#     # print(f"OpenAI: {openai_result}")





In [ ]:
### Задание 2.4: Количественное сравнение точности

# Создаем расширенный набор для тестирования с правильными ответами
test_cases_with_labels = [
    # Сарказм и ирония - должны быть NEGATIVE
    ("Спасибо огромное сотрудникам МТС за то что 3 часа держали меня в очереди! Просто восхитительно 👏", "NEGATIVE"),
    ("Какой замечательный сервис в Пятерочке - касса сломалась прямо передо мной, а персонал даже не извинился", "NEGATIVE"),

    # Смешанные эмоции - должны быть NEUTRAL или зависеть от преобладающего тона
    ("iPhone 13 хороший телефон, но цена кусается. В целом доволен покупкой в re:Store", "NEUTRAL"),
    ("Ресторан Белуга красивый и атмосфера приятная, но официант Максим был невнимателен", "NEUTRAL"),

    # Сложные структуры - требуют понимания контекста
    ("Хотя Tesla Model Y и дорогая машина, и сервис в Рольф Премиум иногда подводит, но в целом я очень доволен покупкой", "POSITIVE"),
    ("Не могу сказать что отель Ритц-Карлтон плохой, просто ожидал большего за такие деньги", "NEUTRAL"),

    # Неформальная речь и сленг
    ("норм телек LG купил в эльдорадо, продавец норм чел был, всё рассказал про функции", "POSITIVE"),
    ("Сходил в кинотеатр Октябрь посмотреть новый фильм Marvel - ну такое себе, но попкорн вкусный был", "NEUTRAL"),

    # Простые случаи для контроля
    ("отличный iphone 14 PRO!!! купил в магазине apple на тверской 😊. Камера супер", "POSITIVE"),
    ("УЖАСНОЕ обслуживание в сбербанке на красной площади.. менеджер иван петров вобще не помог(", "NEGATIVE")
]

# TODO: Рассчитайте точность для каждой модели
# hf_correct = 0
# openai_correct = 0
# total = len(test_cases_with_labels)



In [ ]:
### Задание 2.5: Визуализация сравнения моделей

import matplotlib.pyplot as plt
import numpy as np

# TODO: Создайте визуализацию сравнения точности моделей
# plt.figure(figsize=(12, 8))
# # Создайте графики сравнения

In [ ]:
def create_instruction_dataset(df: pd.DataFrame) -> List[Dict]:
    """
    Создание датасета в формате instruction-following для fine-tuning LLM
    """
    # TODO: Создайте структурированный датасет для fine-tuning LLM
    # Подумайте о структуре instruction-following датасета:
    # - Какие поля должны быть в каждом примере?
    # - Как сформулировать инструкции для модели?
    # - Какие типы задач включить (sentiment, NER, etc.)?
    # - Как структурировать ответы модели?
    #
    # Создайте несколько примеров для разных задач

    pass

# TODO: Протестируйте созданный датасет
# Создайте и проанализируйте instruction dataset
# Выведите примеры в читаемом формате
# Проанализируйте распределение типов задач



### Задание 3.2: Сериализация данных в формате для LLM платформ


In [ ]:
import json

# TODO: Реализуйте сохранение данных в форматах для fine-tuning
# Создайте функции для сохранения данных в форматах:
# 1. JSONL формат для OpenAI fine-tuning API
# 2. CSV формат для общего использования
#
# Изучите требования к форматам:
# - Какая структура нужна для OpenAI fine-tuning?
# - Как правильно структурировать messages?
# - Какие поля обязательны?
#
# Протестируйте сохранение и загрузку данных


